[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/KUMARAGURU-V-S/Lab-experiments/blob/main/GEN-AI-AND-LLM/Experiment-04-Text-Summarization-and-QA/summarization_qa.ipynb)

# summarization_qa.py

In [6]:
import torch
from transformers import BartForConditionalGeneration, BartTokenizer, DistilBertForQuestionAnswering, DistilBertTokenizer

def main():
    # ---------- 1. Text Summarization ----------
    print("--- Text Summarization (BART) ---")

    model_name_bart = "facebook/bart-large-cnn"
    tokenizer_bart = BartTokenizer.from_pretrained(model_name_bart)
    model_bart = BartForConditionalGeneration.from_pretrained(model_name_bart)

    article = (
        "Generative AI refers to a class of artificial intelligence models capable of "
        "producing new content such as text, images, audio, and video. Large Language Models (LLMs) "
        "such as GPT and LLaMA are trained on massive text corpora and can perform a wide range of "
        "natural language tasks including translation, summarization, and question answering. These "
        "models are increasingly being deployed in industry applications ranging from customer support "
        "to software development, transforming how humans interact with machines."
    )

    # Prepare input for summarization
    inputs_bart = tokenizer_bart([article], max_length=1024, return_tensors='pt', truncation=True)

    # Generate summary
    # Use a device (GPU if available, otherwise CPU)
    device = 0 if torch.cuda.is_available() else -1
    if device != -1: # Move model to GPU if available
        model_bart.to(f'cuda:{device}')
        inputs_bart = {k: v.to(f'cuda:{device}') for k, v in inputs_bart.items()}

    summary_ids = model_bart.generate(inputs_bart['input_ids'], num_beams=4, max_length=45, min_length=20, early_stopping=True)
    summary_text = tokenizer_bart.decode(summary_ids[0], skip_special_tokens=True)

    print("Summary:\n", summary_text)
    print()

    # ---------- 2. Question Answering ----------
    print("--- Starting Question Answering (DistilBERT-SQuAD) ---")

    try:
        model_name_qa = "distilbert-base-cased-distilled-squad"
        tokenizer_qa = DistilBertTokenizer.from_pretrained(model_name_qa)
        model_qa = DistilBertForQuestionAnswering.from_pretrained(model_name_qa)

        if device != -1: # Move model to GPU if available
            model_qa.to(f'cuda:{device}')

        context = article
        question = "What are Large Language Models trained on?"

        # Use direct tokenizer call instead of .encode_plus
        inputs_qa = tokenizer_qa(question, context, add_special_tokens=True, return_tensors="pt")
        input_ids = inputs_qa["input_ids"].to(f'cuda:{device}') if device != -1 else inputs_qa["input_ids"]
        attention_mask = inputs_qa["attention_mask"].to(f'cuda:{device}') if device != -1 else inputs_qa["attention_mask"]

        with torch.no_grad():
            outputs = model_qa(input_ids=input_ids, attention_mask=attention_mask)

        answer_start_scores = outputs.start_logits
        answer_end_scores = outputs.end_logits

        answer_start = torch.argmax(answer_start_scores)
        answer_end = torch.argmax(answer_end_scores) + 1

        answer_tokens = input_ids[0, answer_start:answer_end]
        answer = tokenizer_qa.decode(answer_tokens, skip_special_tokens=True)

        print("Question:", question)
        print("Answer:", answer)
        print("--- Question Answering Complete! ---")
    except Exception as e:
        print(f"An error occurred during Question Answering: {e}")

if __name__ == "__main__":
    main()

--- Text Summarization (BART) ---


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

Summary:
 Generative AI refers to a class of artificial intelligence models capable of producing new content. Large Language Models (LLMs) such as GPT and LLaMA are trained on massive text corpora. These models

--- Starting Question Answering (DistilBERT-SQuAD) ---


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

Question: What are Large Language Models trained on?
Answer: massive text corpora
--- Question Answering Complete! ---
